# Automatické tréningy pre rôzne n_input a horizonty (DST+1 … DST+6)

Tento notebook načíta dáta **iba raz** a potom postupne vytvorí a natrénuje samostatný model pre každú kombináciu:
- `n_input` ∈ {6, 12, 18, 24, 30, 36, 42, 48}
- `y_col` = `DST+1` … `DST+6`

Model/weighty sa ukladajú do `models/` a súhrn metrík do `results/summary.csv`.


In [1]:
import os
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from keras.models import Model
from keras.layers import Dense, Input, LSTM, Flatten, TimeDistributed, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Reprodukovateľnosť (voliteľné)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)


2026-04-10 14:14:02.325974: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os; 
print(os.getcwd())

/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/3_modelovanie/train automate


In [3]:
# =========================
# Nastavenia
# =========================

# Cesty k datasetom (ponechané ako v pôvodnom notebooku)
BASE_PATH = "/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/0_datasety/"
file_path_train = BASE_PATH + "train_omni.csv"
file_path_test  = BASE_PATH + "test_omni.csv"


# Tréningové nastavenia
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25  # early stopping na val_mae

# Kombinácie na tréning
#N_INPUT_LIST = [36, 42, 48]
N_INPUT_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
HORIZONS = [3]

# Výstupné priečinky
MODELS_DIR = "models"
RESULTS_DIR = "results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [4]:
# =========================
# Načítanie dát (iba raz)
# =========================
train_raw = pd.read_csv(file_path_train)
test_raw  = pd.read_csv(file_path_test)

# Skontroluj, aké DST+ stĺpce existujú (pomôže odhaliť preklepy v názvoch)
dst_cols = [c for c in train_raw.columns if c.startswith("DST+")]
print("DST+ stĺpce v train:", dst_cols)
print("DST+ stĺpce v test :", [c for c in test_raw.columns if c.startswith("DST+")])

# čas
if "time1" in train_raw.columns:
    train_raw["time1"] = pd.to_datetime(train_raw["time1"])
if "time1" in test_raw.columns:
    test_raw["time1"] = pd.to_datetime(test_raw["time1"])


DST+ stĺpce v train: ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']
DST+ stĺpce v test : ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']


In [5]:
def build_model(n_input: int, n_features: int) -> keras.Model:
    """LSTM model pre multivariačný vstup."""
    inputs = Input(shape=(n_input, n_features))

    x = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
    )(inputs)
    x = LSTM(128, return_sequences=True)(x)
    x = TimeDistributed(Dense(1, activation="linear"))(x)
    x = Flatten()(x)
    outputs = Dense(1, activation="linear")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss="mse", optimizer="adam", metrics=["mae"])
    return model


def make_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, y_col: str, predictors=None):
    """Pripraví train/valid/test dáta pre daný y_col (bez NaN)."""

    if predictors is None:
        predictors = ["DST", "bz_gsm"]   

    features = predictors + [y_col]

    train = train_df[features].copy()
    test = test_df[features].copy()

    # odstránenie NaN
    train = train.dropna().reset_index(drop=True)
    test = test.dropna().reset_index(drop=True)

    # časový split train/valid
    valid_size = int(len(train) * 0.2)

    if valid_size == 0:
        raise ValueError(f"Príliš málo dát po dropna pre {y_col}")

    valid = train.iloc[-valid_size:, :].copy()
    train = train.iloc[:-valid_size, :].copy()

    # X = 2 features
    X_train = train[predictors].values
    y_train = train[y_col].values

    X_val = valid[predictors].values
    y_val = valid[y_col].values

    X_test = test[predictors].values
    y_test = test[y_col].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors

In [6]:
def train_one(y_col: str, n_input: int):
    predictors = ["DST", "bz_gsm"]   

    (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors = make_splits(
        train_raw, test_raw, y_col, predictors=predictors
    )

    train_gen = TimeseriesGenerator(X_train, y_train, length=n_input, batch_size=BATCH_SIZE)
    val_gen   = TimeseriesGenerator(X_val, y_val, length=n_input, batch_size=BATCH_SIZE)
    test_gen  = TimeseriesGenerator(X_test, y_test, length=n_input, batch_size=BATCH_SIZE)

    if len(train_gen) == 0 or len(val_gen) == 0:
        print(f"[SKIP] {y_col}, n_input={n_input} – prázdny generátor")
        return None

    n_features = len(predictors)
    model = build_model(n_input, n_features)

    print("Predictors:", predictors)
    print("X_train shape:", X_train.shape)
    print("First batch X shape:", train_gen[0][0].shape)
    print("Model input shape:", model.input_shape)

    model_path = os.path.join(MODELS_DIR, f"{y_col}_{n_input}H_BZ.keras")
    checkpoint = ModelCheckpoint(model_path, monitor="val_mae", verbose=1, save_best_only=True, mode="min")
    early = EarlyStopping(monitor="val_mae", mode="min", patience=PATIENCE, restore_best_weights=True)
    callbacks = [checkpoint, early]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

    test_loss, test_mae = model.evaluate(test_gen, verbose=0)

    hist_df = pd.DataFrame(history.history)
    hist_csv = os.path.join(RESULTS_DIR, f"history_{y_col}_{n_input}H_BZ.csv")
    hist_df.to_csv(hist_csv, index=False)

    return {
        "y_col": y_col,
        "horizon_hours": int(y_col.split("+")[1]),
        "n_input": n_input,
        "best_val_mae": float(np.min(hist_df["val_mae"])) if "val_mae" in hist_df else np.nan,
        "best_val_loss": float(np.min(hist_df["val_loss"])) if "val_loss" in hist_df else np.nan,
        "test_mae": float(test_mae),
        "test_loss": float(test_loss),
        "model_path": model_path,
        "history_path": hist_csv,
        "epochs_ran": int(len(hist_df)),
    }

In [7]:
# =========================
# Spustenie všetkých tréningov
# =========================
summary_rows = []

for h in HORIZONS:
    y_col = f"DST+{h}"

    # ochrana, ak by stĺpec neexistoval
    if y_col not in train_raw.columns or y_col not in test_raw.columns:
        print(f"[SKIP] {y_col} neexistuje v datasete.")
        continue

    for n_input in N_INPUT_LIST:
        print("\n" + "="*80)
        print(f"Trénujem: y_col={y_col}, n_input={n_input}")
        print("="*80)

        row = train_one(y_col, n_input)
        if row is not None:
            summary_rows.append(row)


summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "summary_3_BZ.csv")
summary.to_csv(summary_path, index=False)

summary.sort_values(["horizon_hours", "n_input"]).head(20), summary_path



Trénujem: y_col=DST+3, n_input=6
Predictors: ['DST', 'bz_gsm']
X_train shape: (229356, 2)
First batch X shape: (256, 6, 2)
Model input shape: (None, 6, 2)
Epoch 1/200


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 269.5644 - mae: 9.2352
Epoch 1: val_mae improved from inf to 7.39956, saving model to models/DST+3_6H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 84s 83ms/step - loss: 269.4518 - mae: 9.2331 - val_loss: 198.0275 - val_mae: 7.3996
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 107.1343 - mae: 6.2597
Epoch 2: val_mae did not improve from 7.39956
896/896 ━━━━━━━━━━━━━━━━━━━━ 78s 87ms/step - loss: 107.1181 - mae: 6.2595 - val_loss: 175.7153 - val_mae: 7.6919
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 85.7503 - mae: 6.0182
Epoch 3: val_mae improved from 7.39956 to 6.96290, saving model to models/DST+3_6H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 76s 85ms/step - loss: 85.7488 - mae: 6.0181 - val_loss: 144.0103 - val_mae: 6.9629
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 82.3807 - mae: 5.8463
Epoch 4: val_mae did not improve from 6.96290
896/896 ━━━━━━━━━━━━━━━━━━━━ 76s 85ms/step - loss: 82.3783 - mae

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 236.7458 - mae: 8.3834
Epoch 1: val_mae improved from inf to 7.34531, saving model to models/DST+3_12H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 154s 160ms/step - loss: 236.6395 - mae: 8.3818 - val_loss: 175.6527 - val_mae: 7.3453
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 96.5518 - mae: 6.0728
Epoch 2: val_mae did not improve from 7.34531
896/896 ━━━━━━━━━━━━━━━━━━━━ 140s 157ms/step - loss: 96.5447 - mae: 6.0728 - val_loss: 173.7931 - val_mae: 8.0804
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - loss: 89.5031 - mae: 5.9894
Epoch 3: val_mae improved from 7.34531 to 7.22202, saving model to models/DST+3_12H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 138s 154ms/step - loss: 89.4959 - mae: 5.9892 - val_loss: 143.7788 - val_mae: 7.2220
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - loss: 86.7715 - mae: 5.8844
Epoch 4: val_mae improved from 7.22202 to 6.70735, saving model to models/DST+3_12H_BZ.keras
896/896

...

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step - loss: 298.2215 - mae: 9.9936
Epoch 1: val_mae improved from inf to 7.80842, saving model to models/DST+3_42H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 492s 534ms/step - loss: 298.0899 - mae: 9.9914 - val_loss: 180.5482 - val_mae: 7.8084
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - loss: 108.5920 - mae: 6.5705
Epoch 2: val_mae improved from 7.80842 to 7.67532, saving model to models/DST+3_42H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 474s 529ms/step - loss: 108.5874 - mae: 6.5704 - val_loss: 168.7141 - val_mae: 7.6753
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 485ms/step - loss: 93.8684 - mae: 6.2009
Epoch 3: val_mae improved from 7.67532 to 7.15229, saving model to models/DST+3_42H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 474s 529ms/step - loss: 93.8696 - mae: 6.2009 - val_loss: 155.5370 - val_mae: 7.1523
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step - loss: 90.7656 - mae: 6.1302
Epoch 4: val_mae did not improve from 7.15229
896/8

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - loss: 249.9481 - mae: 9.2128
Epoch 1: val_mae improved from inf to 7.38228, saving model to models/DST+3_48H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 572s 621ms/step - loss: 249.8376 - mae: 9.2107 - val_loss: 167.3295 - val_mae: 7.3823
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - loss: 93.8244 - mae: 6.1216
Epoch 2: val_mae improved from 7.38228 to 7.10395, saving model to models/DST+3_48H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 553s 617ms/step - loss: 93.8232 - mae: 6.1216 - val_loss: 147.0487 - val_mae: 7.1040
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 565ms/step - loss: 80.4486 - mae: 5.8391
Epoch 3: val_mae improved from 7.10395 to 7.06806, saving model to models/DST+3_48H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 551s 615ms/step - loss: 80.4517 - mae: 5.8391 - val_loss: 141.6119 - val_mae: 7.0681
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - loss: 87.1605 - mae: 5.9312
Epoch 4: val_mae improved from 7.06806 to 6.73194, sa

(   y_col  horizon_hours  n_input  best_val_mae  best_val_loss  test_mae  \
 0  DST+3              3        6      6.636775     113.092819  4.751524   
 1  DST+3              3       12      6.707353     123.321243  4.730121   
 2  DST+3              3       18      6.624518     122.744904  4.654822   
 3  DST+3              3       24      6.591107     122.277496  4.664623   
 4  DST+3              3       30      6.625766     123.980438  4.670840   
 5  DST+3              3       36      6.673523     127.101913  4.657489   
 6  DST+3              3       42      6.627055     124.309021  4.671359   
 7  DST+3              3       48      6.710943     122.650841  4.814578   
 
    test_loss                 model_path                      history_path  \
 0  52.592445   models/DST+3_6H_BZ.keras   results/history_DST+3_6H_BZ.csv   
 1  55.265827  models/DST+3_12H_BZ.keras  results/history_DST+3_12H_BZ.csv   
 2  52.389473  models/DST+3_18H_BZ.keras  results/history_DST+3_18H_BZ.csv   
 3

## Poznámky
- Ak chceš presne poradie ako si písala (najprv `DST+1` pre všetky `n_input`, potom `DST+2`, …), tak to presne robí horný loop (horizonty vonkajší, `n_input` vnútorný).
- Ak chceš opačne (pre dané `n_input` spraviť `DST+1..6`), stačí prehodiť poradie cyklov.
